Notebook to transform sdf to csv. Steps necessary:
1. Write sdf filename
2. Set the partition prefix and the chunk size
3. Read the molecules, compute the Lipinski descriptors and balance the classes
4. Write the partitions - the file names are derived from the rows already on disk,
   so re-running the notebook can neither duplicate a molecule nor invent a row range
5. Audit the directory: every name spans exactly its rows and no molecule is stored twice

In [ ]:
import gzip
import os
import re

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

In [ ]:
path_sdf = os.path.join('..', 'data', '01_raw', 'sdf_files')
filename_sdf = 'Compound_002500001_003000000.sdf.gz'

# NOTE: the row range in every partition file name is derived from the data on
# disk (see write_balanced_partitions below), never from a counter kept in the
# kernel. Re-running this notebook is therefore safe.

In [ ]:
path_csv = os.path.join('..', 'data', '01_raw', 'csv_files')

# Partitions are named <prefix>_<start>_<end> with a 1-based row range.
partition_prefix = 'Balanced'
chunk_size = 10000

In [188]:
sdf_file = os.path.join(path_sdf, filename_sdf)

In [189]:
try:
    with gzip.open(sdf_file, 'rb') as gz:
        supplier = Chem.ForwardSDMolSupplier(gz)
        data = []
        for i, mol in enumerate(supplier):
            if mol is None:
                print(f"Warning: Skipping invalid molecule in at index {i}")
                continue
            try:
                data.append({
                    "SMILES": Chem.MolToSmiles(mol),
                    "Molecular Weight": Descriptors.MolWt(mol),
                    "H-Bond Donors": Chem.Lipinski.NumHDonors(mol),
                    "H-Bond Acceptors": Chem.Lipinski.NumHAcceptors(mol),
                    "LogP": Descriptors.MolLogP(mol),
                })
            except Exception as e:
                print(f"Error processing molecule {i+1}: {e}")

        df = pd.DataFrame(data)
except Exception as e:
    print(f"Error processing file: {e}")

[11:19:01] Explicit valence for atom # 0 Br, 3, is greater than permitted
[11:19:01] ERROR: Could not sanitize molecule ending on line 35204103
[11:19:01] ERROR: Explicit valence for atom # 0 Br, 3, is greater than permitted


[11:19:01] WARNING: not removing hydrogen atom without neighbors
[11:19:01] WARNING: not removing hydrogen atom without neighbors
[11:19:01] WARNING: not removing hydrogen atom without neighbors
[11:19:01] WARNING: not removing hydrogen atom without neighbors
[11:19:08] Explicit valence for atom # 0 Sn, 6, is greater than permitted
[11:19:08] ERROR: Could not sanitize molecule ending on line 36981641
[11:19:08] ERROR: Explicit valence for atom # 0 Sn, 6, is greater than permitted


[11:19:09] WARNING: not removing hydrogen atom without neighbors
[11:19:09] WARNING: not removing hydrogen atom without neighbors
[11:19:26] Explicit valence for atom # 0 Br, 3, is greater than permitted
[11:19:26] ERROR: Could not sanitize molecule ending on line 41391701
[11:19:26] ERROR: Explicit valence for atom # 0 Br, 3, is greater than permitted


[11:20:01] Explicit valence for atom # 1 Cl, 3, is greater than permitted
[11:20:01] ERROR: Could not sanitize molecule ending on line 49640295
[11:20:01] ERROR: Explicit valence for atom # 1 Cl, 3, is greater than permitted
[11:20:01] Explicit valence for atom # 0 Cl, 3, is greater than permitted
[11:20:01] ERROR: Could not sanitize molecule ending on line 49640963
[11:20:01] ERROR: Explicit valence for atom # 0 Cl, 3, is greater than permitted


[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not removing hydrogen atom without neighbors
[11:23:53] WARNING: not r

In [190]:
import numpy as np

In [191]:
def is_lipinski(x: pd.DataFrame) -> pd.DataFrame:
    """Applies Lipinski's Rule of Five to a DataFrame of molecular properties.

    Calculates whether a molecule adheres to at least three of the four main
    Lipinski rules (MW < 500, LogP <= 5, H-Bond Donors <= 5,
    H-Bond Acceptors <= 10). Adds a 'RuleFive' column where 1 indicates
    compliance (passes >= 3 rules) and 0 indicates failure.

    Args:
        x: DataFrame containing molecular properties, including 'Molecular Weight',
           'LogP', 'H-Bond Donors', and 'H-Bond Acceptors'.

    Returns:
        The input DataFrame with an added 'RuleFive' integer column.
    """
    # Lipinski rules
    hdonor = x['H-Bond Donors'] <= 5
    haccept = x['H-Bond Acceptors'] <= 10
    mw = x['Molecular Weight'] < 500
    clogP = x['LogP'] <= 5
    # Apply rules to dataframe
    x['RuleFive'] = np.where(((hdonor & haccept & mw) | (hdonor & haccept & clogP) | (hdonor & mw & clogP) | (haccept & mw & clogP)), 1, 0)
    return x

In [192]:
df= is_lipinski(df)

In [193]:
df.head()

,SMILES,Molecular Weight,H-Bond Donors,H-Bond Acceptors,LogP,RuleFive
0,CCCOc1c(Cl)cc(C(=O)OCC(N)=O)cc1OCC,315.753,1,5,2.1696,1
1,CCCOc1c(Cl)cc(C(=O)OCC(=O)Nc2ccc3c(c2)OCO3)cc1OCC,435.860,1,7,4.0517,1
2,CCCOc1c(Cl)cc(C(=O)OCC(=O)NC2CC2)cc1OCC,355.818,1,5,2.9629,1
3,CCCOc1c(Cl)cc(C(=O)OCC(=O)NCc2ccccc2)cc1OCC,405.878,1,5,4.0007,1
4,CC(C)NC(=O)COC(=O)CN1C(=O)[C@H]2CCCC[C@H]2C1=O,310.350,1,5,0.2294,1


In [194]:
df['RuleFive'].value_counts()

RuleFive
1    423470
0      8867
Name: count, dtype: int64

In [195]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 432337 entries, 0 to 432336
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   SMILES            432337 non-null  object 
 1   Molecular Weight  432337 non-null  float64
 2   H-Bond Donors     432337 non-null  int64  
 3   H-Bond Acceptors  432337 non-null  int64  
 4   LogP              432337 non-null  float64
 5   RuleFive          432337 non-null  int32  
dtypes: float64(2), int32(1), int64(2), object(1)
memory usage: 18.1+ MB


In [ ]:
# One structure can be registered under several PubChem CIDs, so the same
# SMILES may show up more than once. Collapse them before balancing and
# sharding, otherwise the duplicates get a second life in the partitions.
previous_rows = len(df)
df = df.drop_duplicates(subset='SMILES', keep='first').reset_index(drop=True)
print(f'Dropped {previous_rows - len(df)} repeated molecule(s); {len(df)} unique rows remain.')

rulefive_0 = df[df['RuleFive'] == 0]
rulefive_1 = df[df['RuleFive'] == 1]
rulefive_1_equal = rulefive_1.sample(n=len(rulefive_0), random_state=25)
balanced_df = pd.concat([rulefive_1_equal, rulefive_0]).reset_index(drop=True)

In [197]:
rulefive_0.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8867 entries, 32 to 417139
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   SMILES            8867 non-null   object 
 1   Molecular Weight  8867 non-null   float64
 2   H-Bond Donors     8867 non-null   int64  
 3   H-Bond Acceptors  8867 non-null   int64  
 4   LogP              8867 non-null   float64
 5   RuleFive          8867 non-null   int32  
dtypes: float64(2), int32(1), int64(2), object(1)
memory usage: 450.3+ KB


In [198]:
balanced_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17734 entries, 0 to 17733
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   SMILES            17734 non-null  object 
 1   Molecular Weight  17734 non-null  float64
 2   H-Bond Donors     17734 non-null  int64  
 3   H-Bond Acceptors  17734 non-null  int64  
 4   LogP              17734 non-null  float64
 5   RuleFive          17734 non-null  int32  
dtypes: float64(2), int32(1), int64(2), object(1)
memory usage: 762.1+ KB


In [199]:
balanced_df.iloc[42145:42157]

,SMILES,Molecular Weight,H-Bond Donors,H-Bond Acceptors,LogP,RuleFive


In [200]:
balanced_df['RuleFive'].value_counts()

RuleFive
1    8867
0    8867
Name: count, dtype: int64

In [201]:
random_balanced_df = balanced_df.sample(frac=1, random_state=25).reset_index(drop=True)

In [202]:
random_balanced_df['RuleFive'].value_counts()

RuleFive
1    8867
0    8867
Name: count, dtype: int64

In [203]:
random_balanced_df.head()

,SMILES,Molecular Weight,H-Bond Donors,H-Bond Acceptors,LogP,RuleFive
0,Cc1cc2nc3sc(=Cc4ccc(-c5ccc(C(=O)O)cc5)o4)c(=O)...,416.458,1,6,4.03194,1
1,CCOC(=O)C1=C(C)N=c2sc(=Cc3ccc(OC(C)=O)c(OC)c3)...,552.605,0,11,2.74940,0
2,O=[N+]([O-])c1ccccc1CN1CCN(CC2CC3C=CC2C3)CC1,327.428,0,4,2.92460,1
3,CC1=CC2(C)C3C(=O)N(c4cc(Cl)cc(Cl)c4)C(=O)C3C1C...,564.252,0,4,5.80760,0
4,O=c1c2ccccc2nnn1CSc1nc2ccccc2c(=O)n1CCC1=CCCCC1,445.548,0,8,4.14170,1


### Writing the partitions

`data/01_raw/csv_files` is a Kedro `PartitionedDataSet`, so every
`Balanced_<start>_<end>.csv` file is read as its own partition: a molecule that
lands in two files is trained on twice.

Two rules keep that directory consistent, and both are enforced here instead of
being remembered by the kernel:

* **The name describes the data.** `<start>_<end>` is the 1-based range of the
  rows stored in the file, counted from the first row of the dataset: the first
  partition starts at 1, and a short chunk gets a short range - 4,303 rows are
  `Balanced_000080001_000084303.csv`, not `Balanced_000080001_000090000.csv`.
  `scan_partitions` refuses to add anything to a directory whose names do not
  match their contents, and `audit_partitions` re-checks it at the end.
* **Nothing is written twice.** The old loop took its file names from a
  `last_index` counter kept in memory, so re-running it wrote the same rows again
  under new names - that is how four byte-identical copies holding 30,598 rows
  ended up next to the originals. `write_balanced_partitions` derives the next
  range from the rows already stored, skips molecules that are already there,
  refuses to overwrite a partition whose content differs, and writes through a
  `*.partial` file so an interrupted run leaves nothing half-written.

`python rename_partitions.py --apply` (from the project root) renumbers existing
partitions - raw and featurized alike - with the same rule.

In [ ]:
# Partitions are named <prefix>_<start>_<end>, where [start, end] is the 1-based
# range of the rows stored in that file, counted from the first row of the dataset.

def partition_entries(path_csv, prefix):
    """(start, end, name) for every partition on disk, ordered by their range."""
    pattern = re.compile(rf'^{re.escape(prefix)}_(\d{{9}})_(\d{{9}})\.csv$')
    entries = []
    for name in (os.listdir(path_csv) if os.path.isdir(path_csv) else []):
        match = pattern.match(name)
        if match:
            entries.append((int(match.group(1)), int(match.group(2)), name))
    return sorted(entries)


def scan_partitions(path_csv, prefix):
    """Read every stored molecule and verify the file names describe the data.

    Returns:
        (all stored SMILES as one Series, the partition entries).

    Raises:
        RuntimeError: a name does not span exactly its own rows, or the ranges are
            not contiguous from row 1. ``python rename_partitions.py --apply``
            (project root) fixes both.
    """
    entries = partition_entries(path_csv, prefix)
    columns, problems = [], []
    expected_start = 1

    for start, end, name in entries:
        smiles = pd.read_csv(os.path.join(path_csv, name), usecols=['SMILES'])['SMILES']
        correct = f'{prefix}_{expected_start:09d}_{expected_start + len(smiles) - 1:09d}.csv'
        if (start, end) != (expected_start, expected_start + len(smiles) - 1):
            problems.append(f'{name} holds {len(smiles)} row(s), so it should be named {correct}')
        columns.append(smiles)
        expected_start += len(smiles)

    if problems:
        raise RuntimeError(
            'Partition names do not match their contents:\n  ' + '\n  '.join(problems)
            + '\nRun "python rename_partitions.py --apply" from the project root, then re-run this cell.'
        )

    stored = pd.concat(columns, ignore_index=True) if columns else pd.Series(dtype=object)
    return stored, entries


def write_balanced_partitions(frame, path_csv, prefix, chunk_size):
    """Append `frame` as new partitions, named after the rows they really hold.

    Idempotent, unlike the old `last_index` loop:

    * the next range starts after the rows already stored, so
      `<prefix>_<start>_<end>` always spans exactly the rows in the file - a short
      last chunk gets a short range instead of an invented 10,000-row one;
    * molecules that are already stored are never written again;
    * repeats inside `frame` are collapsed first;
    * an existing file with different content raises instead of being overwritten;
    * every file is written to `*.partial` and renamed, so an interrupted run
      cannot leave a half-written partition behind.

    Args:
        frame: Balanced DataFrame with at least a 'SMILES' column.
        path_csv: Directory holding the partitions.
        prefix: File name prefix, e.g. 'Balanced'.
        chunk_size: Maximum rows per partition file.

    Returns:
        The names of the partitions that were written.
    """
    os.makedirs(path_csv, exist_ok=True)

    stored, _entries = scan_partitions(path_csv, prefix)
    rows_on_disk = len(stored)
    already_stored = set(stored)

    unique_rows = frame.drop_duplicates(subset='SMILES', keep='first')
    if len(unique_rows) != len(frame):
        print(f'Dropped {len(frame) - len(unique_rows)} repeated molecule(s) in the incoming frame.')

    new_rows = unique_rows[~unique_rows['SMILES'].isin(already_stored)].reset_index(drop=True)
    if len(new_rows) != len(unique_rows):
        print(f'Skipped {len(unique_rows) - len(new_rows)} molecule(s) that are already stored.')
    if new_rows.empty:
        print(f'Nothing to write: all {rows_on_disk} stored molecule(s) are already on disk.')
        return []

    written = []
    for offset in range(0, len(new_rows), chunk_size):
        chunk = new_rows.iloc[offset:offset + chunk_size]
        # The range spans the rows in this file, whatever chunk_size allowed.
        start = rows_on_disk + offset + 1
        end = start + len(chunk) - 1
        filename = f'{prefix}_{start:09d}_{end:09d}.csv'
        target = os.path.join(path_csv, filename)

        if os.path.exists(target):
            on_disk = pd.read_csv(target, usecols=['SMILES'])['SMILES'].tolist()
            if on_disk == chunk['SMILES'].tolist():
                print(f'{filename} already holds these rows, leaving it alone.')
                continue
            raise RuntimeError(
                f'{filename} already exists with different content. Delete it, or run '
                f'"python clean_dataset.py --apply" from the project root, before writing again.'
            )

        partial = target + '.partial'
        chunk.to_csv(partial, index=False)
        os.replace(partial, target)
        written.append(filename)

    print(f'Wrote {len(written)} new partition(s) covering {len(new_rows)} row(s).')
    return written

In [ ]:
written_partitions = write_balanced_partitions(
    random_balanced_df,
    path_csv,
    prefix=partition_prefix,
    chunk_size=chunk_size,
)
written_partitions

In [ ]:
def audit_partitions(path_csv, prefix):
    """Fail loudly if a molecule is stored twice or a name does not match its rows."""
    stored, entries = scan_partitions(path_csv, prefix)  # raises when the names are wrong

    counts = stored.value_counts()
    repeated = counts[counts > 1]
    print(
        f'{len(stored)} row(s) in {len(entries)} partition(s), {counts.size} distinct molecule(s), '
        f'{len(repeated)} stored more than once.'
    )
    if not repeated.empty:
        raise AssertionError(
            f'{len(repeated)} molecule(s) are stored more than once - run '
            f'"python clean_dataset.py --apply" from the project root to remove them.'
        )
    return counts.size


audit_partitions(path_csv, partition_prefix)